<a href="https://colab.research.google.com/github/sonjoy1s/Kaggle_Deep_Lr/blob/main/Lung_Cancer2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import pandas as pd

In [39]:
df = pd.read_csv("/content/survey lung cancer (1).csv")
df.head()

,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,M,69,1,2,2,1,1,2,1,2,2,2,2,2,2,YES
1,M,74,2,1,1,1,2,2,2,1,1,1,2,2,2,YES
2,F,59,1,1,1,2,1,2,1,2,1,2,2,1,2,NO
3,M,63,2,2,2,1,1,1,1,1,2,1,1,2,2,NO
4,F,63,1,2,1,1,1,1,1,2,1,2,2,1,1,NO


In [40]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['GENDER'] = le.fit_transform(df['GENDER'])
df['LUNG_CANCER'] = le.fit_transform(df['LUNG_CANCER'])

In [41]:
df.head()

,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,1,69,1,2,2,1,1,2,1,2,2,2,2,2,2,1
1,1,74,2,1,1,1,2,2,2,1,1,1,2,2,2,1
2,0,59,1,1,1,2,1,2,1,2,1,2,2,1,2,0
3,1,63,2,2,2,1,1,1,1,1,2,1,1,2,2,0
4,0,63,1,2,1,1,1,1,1,2,1,2,2,1,1,0


In [42]:
target = 'LUNG_CANCER'
X = df.drop(columns=[target])
y = df[target]

In [43]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

In [44]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [45]:
import numpy as np
import pandas as pd
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [46]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    #convet to pytroch tensor
    self.features = torch.tensor(features,dtype=torch.float32)
    self.labels = torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)


  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [47]:
#train dataset and test dataset make :

train_dataset = CustomDataset(X_train, y_train.values)

test_dataset = CustomDataset(X_test, y_test.values)

In [48]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [49]:
class MyNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(num_features,64),
        nn.ReLU(),
        nn.Linear(64,1),
        nn.Sigmoid()

    )


  def forward(self,X):
    return self.model(X)

In [50]:
epochs = 100
learning_rate = 0.001

In [51]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MyNN(X_train.shape[1])
model = model.to(device)

In [52]:
#loss function
criterion = nn.BCELoss()
#optimizer
optimizer = optim.Adam(model.parameters(),lr=learning_rate)

In [53]:
for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features,batch_labels in train_loader:
    #move to gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # Reshape and convert labels to float to match model output for BCEWithLogitsLoss
    # Assuming the model's final layer will be adjusted to output 1 dimension
    batch_labels = batch_labels.float().unsqueeze(1)

    #forward pass
    output = model(batch_features)

    #calculate loss
    loss = criterion(output,batch_labels)

    #back pass
    optimizer.zero_grad()  # 1st e optimizer 0 kore nite hoy backward er jonno
    loss.backward()

    #update
    optimizer.step()

    # print(loss.item())
    total_epoch_loss = total_epoch_loss + loss.item()


avg_loss = total_epoch_loss / len(train_loader)

print(avg_loss)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


0.09611862432211637


In [54]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        predicted = (outputs > 0.5).float()  # Convert probabilities to binary predictions

        total += batch_labels.size(0)
        correct += (predicted.squeeze() == batch_labels.float()).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the test set: {accuracy:.3f}%')

Accuracy on the test set: 92.308%
